# Early FTUE analysis

**Purpose:** 

**Data range:** 

---



In [1]:
# hide-output

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np
import plotly.graph_objects as go

bqc = BigQueryConnector()

## Aux functions

In [46]:
# hide-output
def compute_weighted_progression(data, measure_col, dimension_cols=['install_dt', 'days_since_install'], min_bucket_size=50):
    """
    Compute weighted average progression metric for a given measure across dimensions.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Source data containing user_id, the measure column, and dimension columns
    measure_col : str
        Column name to compute weighted average for (e.g., 'max_level', 'max_gameday')
    dimension_cols : list
        Dimensions to group by (default: ['install_dt', 'days_since_install'])
    min_bucket_size : int
        Minimum users per bucket to include (default: 50)
    
    Returns:
    --------
    pd.DataFrame
        Aggregated data with weighted average and cohort user counts
    """
    
    # Step 1: Count unique users per dimension + measure bucket
    agg = data.groupby(dimension_cols + [measure_col]).agg(
        unique_users=('user_id', 'nunique')
    ).reset_index()
    
    # Step 2: Total unique users per dimension combination
    dimension_total_users = data.groupby(dimension_cols).agg(
        total_unique_users=('user_id', 'nunique')
    ).reset_index()
    
    # Step 3: Merge and compute percentage share
    agg = agg.merge(dimension_total_users, on=dimension_cols)
    agg['percentage_of_users'] = agg['unique_users'] / agg['total_unique_users']
    
    # Step 4: Compute weighted average
    weighted_avg = agg.groupby(dimension_cols, group_keys=False).apply(
        lambda x: (x[measure_col] * x['unique_users']).sum() / x['unique_users'].sum(),
        include_groups=False
    ).reset_index()
    
    weighted_avg.columns = dimension_cols + [f'weighted_avg_{measure_col}']
    agg = agg.merge(weighted_avg, on=dimension_cols)
    
    # Step 5: Drop small buckets
    agg = agg[agg['unique_users'] >= min_bucket_size]
    
    # Step 6: Collapse to one row per dimension combination
    agg = agg.groupby(dimension_cols).agg(
        cohort_users=('unique_users', 'sum'),
        **{f'weighted_avg_{measure_col}': ('weighted_avg_' + measure_col, 'first')}
    ).reset_index()
    
    return agg

def weighted_quantiles(group, quantiles=[0.1, 0.5, 0.9], measure_col='max_level'):
    """Return P10 / P50 / P90 of measure_col, using user counts as weights.

    Sorts by the measure, accumulates weights, then uses searchsorted to find
    the value at each quantile threshold — equivalent to a weighted percentile.
    """
    levels = group[measure_col].values
    weights = group['users'].values
    sorted_idx = np.argsort(levels)
    levels, weights = levels[sorted_idx], weights[sorted_idx]
    cum_weights = np.cumsum(weights)
    total = cum_weights[-1]
    result = {}
    for q in quantiles:
        idx = np.searchsorted(cum_weights, q * total)
        result[f'p{int(q * 100)}'] = levels[min(idx, len(levels) - 1)]
    return pd.Series(result)


def add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=None, show_annotations=True):
    if not show_annotations:
        return fig
    
    subplot_axes = set()
    all_y_vals = []

    for trace in fig.data:
        xaxis = getattr(trace, 'xaxis', None) or 'x'
        yaxis = getattr(trace, 'yaxis', None) or 'y'
        subplot_axes.add((xaxis, yaxis))
        if trace.y is not None:
            for v in trace.y:
                try:
                    val = float(v)
                    if not np.isnan(val):
                        all_y_vals.append(val)
                except (TypeError, ValueError):
                    pass

    global_y_min = min(all_y_vals) if all_y_vals else 0.0
    global_y_max = max(all_y_vals) if all_y_vals else 1.0
    y_bottom = min(0.0, global_y_min)
    y_top = global_y_max + abs(global_y_max - y_bottom) * 0.05

    ftue_flags_in_data = set()
    if data is not None and ftue_col in data.columns:
        ftue_flags_in_data = set(data[ftue_col].unique())

    for ftue_type, events in events_config.items():
        if ftue_flags_in_data and ftue_type not in ftue_flags_in_data:
            continue
            
        for event in events:
            for (xaxis, yaxis) in subplot_axes:
                if ftue_type == 'B.new':
                    text_list = ['', event['name']]
                    text_pos = 'top center'
                else:
                    text_list = [event['name'], '']
                    text_pos = 'top center'
                fig.add_trace(go.Scatter(
                    x=[event['level'], event['level']],
                    y=[y_bottom, y_top],
                    mode='lines+text',
                    line=dict(color=event['color'], dash='dash', width=1),
                    opacity=0.7,
                    legendgroup=ftue_type,
                    showlegend=False,
                    text=text_list,
                    textposition=text_pos,
                    xaxis=xaxis,
                    yaxis=yaxis,
                    hoverinfo='skip',
                ))

    return fig


from plotly.subplots import make_subplots

def plot_percentile_comparison(level_pcts_by_group, percentile='all'):
    """
    Compare A.old vs B.new level percentiles by days since install.

    percentile: 'p10', 'p50', 'p90' — single metric line chart + B.new-minus-A.old diff bar
                'all'               — band chart with P10-P90 shaded region and P50 median line
    """
    df = level_pcts_by_group.sort_values('days_since_install')
    platforms = sorted(df['platform'].unique())

    COLORS = {
        'A.old': {'solid': 'rgba(59,130,246,1)',  'band': 'rgba(59,130,246,0.15)'},
        'B.new': {'solid': 'rgba(239,68,68,1)',   'band': 'rgba(239,68,68,0.15)'},
    }

    if percentile == 'all':
        fig = make_subplots(
            rows=1, cols=len(platforms),
            subplot_titles=platforms,
            shared_yaxes=True,
        )

        for col_idx, platform in enumerate(platforms, 1):
            pdf = df[df['platform'] == platform]
            show_legend = col_idx == 1

            for ftue_flag, c in COLORS.items():
                gdf = pdf[pdf['FTUE_flag'] == ftue_flag].sort_values('days_since_install')

                # P10 — invisible, anchor for fill
                fig.add_trace(go.Scatter(
                    x=gdf['days_since_install'], y=gdf['p10_max_level'],
                    mode='lines', line=dict(width=0),
                    showlegend=False, legendgroup=ftue_flag,
                    hovertemplate='P10: %{y:.0f}<extra></extra>',
                ), row=1, col=col_idx)

                # P90 — fills down to P10
                fig.add_trace(go.Scatter(
                    x=gdf['days_since_install'], y=gdf['p90_max_level'],
                    mode='lines', line=dict(width=0),
                    fill='tonexty', fillcolor=c['band'],
                    name=f'{ftue_flag} P10–P90', legendgroup=ftue_flag,
                    showlegend=show_legend,
                    hovertemplate='P90: %{y:.0f}<extra></extra>',
                ), row=1, col=col_idx)

                # P50 — solid median line
                fig.add_trace(go.Scatter(
                    x=gdf['days_since_install'], y=gdf['p50_max_level'],
                    mode='lines+markers', line=dict(width=2.5, color=c['solid']),
                    name=f'{ftue_flag} P50 (median)', legendgroup=ftue_flag,
                    showlegend=show_legend,
                    hovertemplate='P50: %{y:.0f}<extra></extra>',
                ), row=1, col=col_idx)

        fig.update_layout(
            title='Player level P10 / P50 / P90 by days since install: A.old vs B.new',
            width=1200, height=500,
        )
        fig.update_xaxes(title_text='Days since install')
        fig.update_yaxes(title_text='Max level', col=1)
        fig.show()

    else:
        col = f'{percentile}_max_level'

        pivot = df.pivot_table(
            index=['days_since_install', 'platform'],
            columns='FTUE_flag',
            values=col,
        ).reset_index()
        pivot['diff'] = pivot['B.new'] - pivot['A.old']
        pivot['pct_diff'] = (pivot['diff'] / pivot['A.old'] * 100).round(1)

        fig1 = px.line(
            df, x='days_since_install', y=col,
            color='FTUE_flag', facet_col='platform',
            markers=True,
            color_discrete_map={'A.old': 'rgba(59,130,246,1)', 'B.new': 'rgba(239,68,68,1)'},
            title=f'{percentile.upper()} Level Progression: A.old vs B.new',
            width=1200, height=500,
            labels={'days_since_install': 'Days since install', col: 'Max level'},
        )
        fig1.show()

        fig2 = px.bar(
            pivot, x='days_since_install', y='diff',
            color='platform', facet_row='platform', barmode='group',
            title=f'{percentile.upper()} Level Difference (B.new − A.old)',
            width=1200, height=600,
            hover_data={'diff': ':.2f', 'pct_diff': True, 'A.old': True, 'B.new': True},
            labels={'diff': 'Level difference', 'days_since_install': 'Days since install'},
        )
        fig2.show()

## Get data

### Player level and game day

In [3]:
# calculate the start date based on the number of days from 2026-06-01 to today

import datetime as dt

new_ftue_date = dt.datetime(2026, 6, 1)
days_from_start = (dt.datetime.today() - new_ftue_date).days
start_date1 = new_ftue_date - dt.timedelta(days=days_from_start)
end_date1 = new_ftue_date-dt.timedelta(days=1)
start_date2 = new_ftue_date
end_date2 = dt.datetime.today()-dt.timedelta(days=1)

# print all dates
print(f"Start Date 1: {start_date1.strftime('%Y-%m-%d')}")
print(f"End Date 1: {end_date1.strftime('%Y-%m-%d')}")
print(f"Start Date 2: {start_date2.strftime('%Y-%m-%d')}")
print(f"End Date 2: {end_date2.strftime('%Y-%m-%d')}")

Start Date 1: 2026-05-09
End Date 1: 2026-05-31
Start Date 2: 2026-06-01
End Date 2: 2026-06-23


In [4]:
# hide-output
refresh_data = True

In [5]:
# hide-output
# Point to the SQL file and set the start date for the cohort window

query_location = './sql/playerlevel.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 17.45 GB when run.
Estimated query cost: $0.12


In [6]:
# hide-output
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [7]:
# hide-output
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,acquisition_type,install_build_version
0,3EDE3E8F52C1C70F,2026-05-28,2026-05-28,2026-05-24,2026-05-01,0,3,2,AND,Non-Attributed,0.75.0
1,EDE1EE1A41466EB1,2026-05-29,2026-05-28,2026-05-24,2026-05-01,1,7,4,AND,CPE,0.75.0
2,F3D2861FAD19BF7C,2026-05-30,2026-05-28,2026-05-24,2026-05-01,2,7,4,AND,CPE,0.75.0
3,9709475D46C7F441,2026-05-29,2026-05-29,2026-05-24,2026-05-01,0,4,2,AND,CPE,0.75.0
4,603C7EF87DBCFA08,2026-05-31,2026-05-29,2026-05-24,2026-05-01,2,5,3,AND,Non-Attributed,0.75.0
...,...,...,...,...,...,...,...,...,...,...,...
170968,BE255BCA34CA8F34,2026-06-23,2026-06-23,2026-06-21,2026-06-01,0,5,3,AND,CPE,0.78.0
170969,D398FF72EE7409AE,2026-06-23,2026-06-23,2026-06-21,2026-06-01,0,4,2,IOS,Non-Attributed,0.78.0
170970,CAFF7D42E5A2438C,2026-06-23,2026-06-23,2026-06-21,2026-06-01,0,6,3,IOS,CPE,0.78.0
170971,1F49FEA27E5A9A61,2026-06-23,2026-06-23,2026-06-21,2026-06-01,0,2,1,AND,CPE,0.77.0


### Retention

In [8]:
# hide-output
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/retention.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.29 GB when run.
Estimated query cost: $0.01


In [9]:
# hide-output
retention_data = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data = bqc.get(query='./sql/retention.sql', is_path=True, query_parameters=parameters)
    retention_data.to_pickle('./data/retention.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data = pd.read_pickle('./data/retention.pkl')

In [10]:
# hide-output
retention_data.sort_values(['install_dt', 'dx','platform'], inplace=True)
retention_data[retention_data['platform'] == 'AND']

,install_dt,dx,platform,cohort_size,retained_size,retention_rate
0,2026-05-09,0,AND,375,375,1.000000
2,2026-05-09,1,AND,375,153,0.408000
4,2026-05-09,3,AND,375,90,0.240000
7,2026-05-09,7,AND,375,77,0.205333
9,2026-05-09,14,AND,375,60,0.160000
...,...,...,...,...,...,...
359,2026-06-21,0,AND,478,478,1.000000
360,2026-06-21,1,AND,478,195,0.407950
362,2026-06-22,0,AND,487,487,1.000000
364,2026-06-22,1,AND,487,187,0.383984


In [11]:
# hide-output
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/retentiontotal.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.29 GB when run.
Estimated query cost: $0.01


In [12]:
# hide-output
retention_data_total = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data_total = bqc.get(query='./sql/retentiontotal.sql', is_path=True, query_parameters=parameters)
    retention_data_total.to_pickle('./data/retentiontotal.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total = pd.read_pickle('./data/retentiontotal.pkl')

In [13]:
# hide-output
retention_data_total.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total[retention_data_total['dx'] == 14]

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
17,14,AND,A.Pre-FTUE revamp,9,3305,545,0.164902
16,14,AND,B.Post-FTUE revamp,9,4240,621,0.146462
19,14,IOS,A.Pre-FTUE revamp,9,7936,1142,0.143901
18,14,IOS,B.Post-FTUE revamp,9,7503,1209,0.161136


In [14]:
# hide-output
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/retentiontotalNA.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.29 GB when run.
Estimated query cost: $0.01


In [15]:
# hide-output
retention_data_total_na = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    retention_data_total_na = bqc.get(query='./sql/retentiontotalNA.sql', is_path=True, query_parameters=parameters)
    retention_data_total_na.to_pickle('./data/retentiontotalNA.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total_na = pd.read_pickle('./data/retentiontotalNA.pkl')

In [16]:
# hide-output
retention_data_total_na.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total_na

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,23,3926,3926,1.000000
0,0,AND,B.Post-FTUE revamp,23,5624,5624,1.000000
3,0,IOS,A.Pre-FTUE revamp,23,8208,8208,1.000000
2,0,IOS,B.Post-FTUE revamp,23,11229,11229,1.000000
5,1,AND,A.Pre-FTUE revamp,22,3876,1255,0.323787
4,1,AND,B.Post-FTUE revamp,22,5340,1590,0.297753
7,1,IOS,A.Pre-FTUE revamp,22,8164,2735,0.335007
6,1,IOS,B.Post-FTUE revamp,22,10610,3458,0.325919
9,3,AND,A.Pre-FTUE revamp,20,3788,752,0.198522
8,3,AND,B.Post-FTUE revamp,20,4948,824,0.166532


## Process data

In [17]:
# hide-output
dt_mode = 'install_dt'

data['install_dt'] = data[dt_mode]

data['CPE_flag'] = ['Y' if x == 'CPE' else 'N' for x in data['acquisition_type']]

data['FTUE_flag'] = ['B.new' if x >='0.76.0' else 'A.old' for x in data['install_build_version']]

# Making comparison fair
max_dayx_B_new = (pd.to_datetime('today') - pd.to_datetime('2026-06-01')).days
data = data[~(data['days_since_install'] > max_dayx_B_new)]

data.loc[:,'dummy'] = 'dummy'

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
min_days_since_install = 0
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
data = data[data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(data['install_dt'])).dt.days - min_days_since_install]

data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,3EDE3E8F52C1C70F,2026-05-28,2026-05-28,2026-05-24,2026-05-01,0,3,2,AND,Non-Attributed,0.75.0,N,A.old,dummy
1,EDE1EE1A41466EB1,2026-05-29,2026-05-28,2026-05-24,2026-05-01,1,7,4,AND,CPE,0.75.0,Y,A.old,dummy
2,F3D2861FAD19BF7C,2026-05-30,2026-05-28,2026-05-24,2026-05-01,2,7,4,AND,CPE,0.75.0,Y,A.old,dummy
3,9709475D46C7F441,2026-05-29,2026-05-29,2026-05-24,2026-05-01,0,4,2,AND,CPE,0.75.0,Y,A.old,dummy
4,603C7EF87DBCFA08,2026-05-31,2026-05-29,2026-05-24,2026-05-01,2,5,3,AND,Non-Attributed,0.75.0,N,A.old,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170968,BE255BCA34CA8F34,2026-06-23,2026-06-23,2026-06-21,2026-06-01,0,5,3,AND,CPE,0.78.0,Y,B.new,dummy
170969,D398FF72EE7409AE,2026-06-23,2026-06-23,2026-06-21,2026-06-01,0,4,2,IOS,Non-Attributed,0.78.0,N,B.new,dummy
170970,CAFF7D42E5A2438C,2026-06-23,2026-06-23,2026-06-21,2026-06-01,0,6,3,IOS,CPE,0.78.0,Y,B.new,dummy
170971,1F49FEA27E5A9A61,2026-06-23,2026-06-23,2026-06-21,2026-06-01,0,2,1,AND,CPE,0.77.0,Y,B.new,dummy


In [18]:
# hide-output
test = data.groupby(['FTUE_flag']).agg(
    users=('user_id', 'nunique')
).reset_index()

test

,FTUE_flag,users
0,A.old,16877
1,B.new,21775


## Retention

Retention curve: the share of a cohort's total users who were active on each calendar day since install. Plotted on a **log scale** so that differences between cohorts remain visible at longer horizons where absolute percentages are very small. Apr 2024 D1 retention was ~55%; Apr 2026 has declined to ~51%.

In [19]:
# hide-output
retention_data['combined_dimension'] = retention_data['dx'].astype(str) + '_' + retention_data['platform']
retention_data


,install_dt,dx,platform,cohort_size,retained_size,retention_rate,combined_dimension
0,2026-05-09,0,AND,375,375,1.000000,0_AND
1,2026-05-09,0,IOS,900,900,1.000000,0_IOS
2,2026-05-09,1,AND,375,153,0.408000,1_AND
3,2026-05-09,1,IOS,900,347,0.385556,1_IOS
4,2026-05-09,3,AND,375,90,0.240000,3_AND
...,...,...,...,...,...,...,...
363,2026-06-22,0,IOS,974,974,1.000000,0_IOS
364,2026-06-22,1,AND,487,187,0.383984,1_AND
365,2026-06-22,1,IOS,974,394,0.404517,1_IOS
367,2026-06-23,0,AND,468,468,1.000000,0_AND


In [20]:

fig = px.line(retention_data[retention_data['dx'] !=0], 
              x='install_dt', 
              y='retention_rate',
              color='combined_dimension',
              title='Retention rate',
              facet_row='platform',
              width=1200,
              height=800,
              hover_data={'install_dt': True, 'retained_size': True},)

#fig.show()

In [21]:
# hide-output
retention_data_total

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,23,7258,7258,1.000000
0,0,AND,B.Post-FTUE revamp,23,9980,9980,1.000000
3,0,IOS,A.Pre-FTUE revamp,23,15104,15104,1.000000
2,0,IOS,B.Post-FTUE revamp,23,19729,19729,1.000000
5,1,AND,A.Pre-FTUE revamp,22,7180,2840,0.395543
4,1,AND,B.Post-FTUE revamp,22,9430,3675,0.389714
7,1,IOS,A.Pre-FTUE revamp,22,15058,6279,0.416988
6,1,IOS,B.Post-FTUE revamp,22,18751,7727,0.412085
9,3,AND,A.Pre-FTUE revamp,20,6990,1950,0.278970
8,3,AND,B.Post-FTUE revamp,20,8488,2209,0.260250


### Cohort sizes

In [22]:
# Prepare data
df_plot = retention_data_total[retention_data_total['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='cohort_size',
    color='FTUE_flag',
    text='cohort_size',
    title='Cohort sizes',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:0}', textposition='outside')
fig.update_layout(
    #yaxis_tickformat='.0%',
    #yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

### Retantion rate 

In [23]:
# Prepare data
df_plot = retention_data_total[retention_data_total['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='retention_rate',
    color='FTUE_flag',
    text='retention_rate',
    title='Retention rate by FTUE group (95% CI based on cohort size)',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.update_layout(
    yaxis_tickformat='.0%',
    yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

### Retention rate for Organics

In [24]:
# Prepare data
df_plot = retention_data_total_na[retention_data_total_na['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='retention_rate',
    color='FTUE_flag',
    text='retention_rate',
    title='Retention rate by FTUE group (Only non-attributed)',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.update_layout(
    yaxis_tickformat='.0%',
    yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

## Player max level distribution

In [33]:
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
111122,10007491E7935123,2026-06-06,2026-06-06,2026-05-31,2026-06-01,0,6,3,IOS,CPE,0.76.0,Y,B.new,dummy
111292,10007491E7935123,2026-06-07,2026-06-06,2026-05-31,2026-06-01,1,6,4,IOS,CPE,0.76.0,Y,B.new,dummy
111995,10007491E7935123,2026-06-08,2026-06-06,2026-05-31,2026-06-01,2,7,4,IOS,CPE,0.76.0,Y,B.new,dummy
114190,10007491E7935123,2026-06-09,2026-06-06,2026-05-31,2026-06-01,3,7,4,IOS,CPE,0.76.0,Y,B.new,dummy
114354,10007491E7935123,2026-06-10,2026-06-06,2026-05-31,2026-06-01,4,8,5,IOS,CPE,0.76.0,Y,B.new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40492,FFFE079DACB6000A,2026-05-17,2026-05-16,2026-05-10,2026-05-01,1,6,4,IOS,UA,0.75.0,N,A.old,dummy
44255,FFFE079DACB6000A,2026-05-19,2026-05-16,2026-05-10,2026-05-01,3,8,5,IOS,UA,0.75.0,N,A.old,dummy
42074,FFFE079DACB6000A,2026-05-20,2026-05-16,2026-05-10,2026-05-01,4,8,5,IOS,UA,0.75.0,N,A.old,dummy
44312,FFFE079DACB6000A,2026-05-22,2026-05-16,2026-05-10,2026-05-01,6,8,5,IOS,UA,0.75.0,N,A.old,dummy


In [36]:
# hide-output
pl_ftue_funnel_agg = data.groupby(['max_level','FTUE_flag','platform', 'days_since_install']).agg(
    users=('user_id', 'nunique')
).reset_index()

pl_ftue_funnel_total_agg = pl_ftue_funnel_agg.groupby(['FTUE_flag','platform','days_since_install']).agg(
    total_users=('users', 'sum')
).reset_index()

pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(pl_ftue_funnel_total_agg, on=['FTUE_flag','platform','days_since_install'])
pl_ftue_funnel_agg['pctg_users'] = pl_ftue_funnel_agg['users'] / pl_ftue_funnel_agg['total_users']


pl_ftue_funnel_agg['pctg_diff_users'] = pl_ftue_funnel_agg.groupby(['max_level','platform','days_since_install'])['pctg_users'].pct_change().fillna(0)

pl_ftue_funnel_agg['combined_dimension'] = pl_ftue_funnel_agg['FTUE_flag'].astype(str) + ' | ' + pl_ftue_funnel_agg['days_since_install'].astype(str)

# Calculate percentiles by FTUE_flag, platform, and days_since_install
level_dist = data.groupby(['days_since_install', 'max_level', 'FTUE_flag', 'platform']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts_by_group = level_dist.groupby(['days_since_install', 'FTUE_flag', 'platform']).apply(
    weighted_quantiles, measure_col='max_level', include_groups=False
).reset_index()

# Rename columns for clarity
level_pcts_by_group = level_pcts_by_group.rename(columns={'p10': 'p10_max_level', 'p50': 'p50_max_level', 'p90': 'p90_max_level'})

# Merge into pl_ftue_funnel_agg
pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(
    level_pcts_by_group, 
    on=['days_since_install', 'FTUE_flag', 'platform'], 
    how='left'
)

pl_ftue_funnel_agg = pl_ftue_funnel_agg.loc[pl_ftue_funnel_agg['days_since_install'].isin([0,1,3,7,14,21])]


pl_ftue_funnel_agg

,max_level,FTUE_flag,platform,days_since_install,users,total_users,pctg_users,pctg_diff_users,combined_dimension,p10_max_level,p50_max_level,p90_max_level
0,1,A.old,AND,0,603,5109,0.118027,0.0,A.old | 0,1,4,6
1,1,A.old,AND,1,61,2707,0.022534,0.0,A.old | 1,3,6,8
3,1,A.old,AND,3,24,1896,0.012658,0.0,A.old | 3,5,8,11
7,1,A.old,AND,7,4,1210,0.003306,0.0,A.old | 7,6,11,15
14,1,A.old,AND,14,1,536,0.001866,0.0,A.old | 14,8,14,19
...,...,...,...,...,...,...,...,...,...,...,...,...
2741,154,B.new,AND,3,2,2151,0.000930,0.0,B.new | 3,5,8,11
2745,154,B.new,AND,7,1,1352,0.000740,0.0,B.new | 7,6,11,15
2752,154,B.new,AND,14,1,610,0.001639,0.0,B.new | 14,7,13,20
2757,155,B.new,AND,3,1,2151,0.000465,0.0,B.new | 3,5,8,11


In [37]:
# hide-output
# Define event parameters for level annotations
events_config = {
    'A.old': [
        {'level': 7, 'name': 'SP', 'color':'blue'},
        {'level': 8, 'name': 'Deco', 'color': 'blue'},
        {'level': 10, 'name': 'TimedC', 'color': 'blue'},
        {'level': 16, 'name': 'TA', 'color': 'blue'},
        {'level': 20, 'name': 'GenB', 'color': 'blue'},
        {'level': 25, 'name': 'TgtEvt', 'color': 'blue'},
    ],
    'B.new': [
        {'level': 6, 'name': 'SP', 'color': 'red'},
        {'level': 9, 'name': 'TASign', 'color': 'red'},
        {'level': 10, 'name': 'Deco', 'color': 'red'},
        {'level': 12, 'name': 'TA', 'color': 'red'},
        {'level': 14, 'name': 'GenB', 'color': 'red'},
        {'level': 17, 'name': 'TimedC', 'color': 'red'},
        {'level': 20, 'name': 'TgtEvt', 'color': 'red'},
    ]
}


In [38]:
def add_median_lines(fig, level_pcts_by_group, x_col='p50_max_level', ftue_col='FTUE_flag', platform_col='platform'):
    """
    Add vertical dotted lines at median (P50) values to a plotly figure.
    
    Parameters:
    -----------
    fig : plotly.graph_objs._figure.Figure
        The figure to add lines to
    level_pcts_by_group : pd.DataFrame
        DataFrame containing percentile data with FTUE_flag and platform columns
    x_col : str
        Column name for the x-axis values (default: 'p50_max_level')
    ftue_col : str
        Column name for FTUE grouping (default: 'FTUE_flag')
    platform_col : str
        Column name for platform grouping (default: 'platform')
    
    Returns:
    --------
    plotly.graph_objs._figure.Figure
        Figure with added median lines
    """
    
    # Get unique platforms and map to xaxis
    platforms = sorted(pl_ftue_funnel_agg[platform_col].unique())
    platform_to_xaxis = {platform: f"x{i+1}" if i > 0 else "x" for i, platform in enumerate(platforms)}
    
    # Get unique FTUE flags and their colors
    color_map = {'A.old': 'blue', 'B.new': 'red'}
    
    # Get P50 values per FTUE_flag and platform from pl_ftue_funnel_agg
    medians = pl_ftue_funnel_agg[['FTUE_flag', 'platform', 'p50_max_level']].drop_duplicates()
    
    for _, row in medians.iterrows():
        ftue_flag = row['FTUE_flag']
        platform = row['platform']
        median_val = row['p50_max_level']
        xaxis = platform_to_xaxis.get(platform, 'x')
        
        fig.add_shape(
            type="line",
            x0=median_val, x1=median_val,
            y0=0, y1=1,
            yref="paper",
            xref=xaxis,
            line=dict(color=color_map.get(ftue_flag, 'gray'), dash="dot", width=2),
            opacity=0.7,
        )
    
    return fig


In [39]:
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='users',
              color='combined_dimension',
              title='Players level distribution',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)
#fig = add_median_lines(fig, pl_ftue_funnel_agg, x_col='p50_max_level', ftue_col='FTUE_flag', platform_col='platform')

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='pctg_users',
              color='combined_dimension',
              title='Players at each level (percentage)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='pctg_diff_users',
              color='combined_dimension',
              title='Players at each level (percentage diff change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)

fig.show()

### Percentile comparison

In [47]:
plot_percentile_comparison(level_pcts_by_group, percentile='all')

# To inspect a single percentile with diff bar:
# plot_percentile_comparison(level_pcts_by_group, percentile='p50')
# plot_percentile_comparison(level_pcts_by_group, percentile='p10')
# plot_percentile_comparison(level_pcts_by_group, percentile='p90')

### Weighted average

For each install cohort, tracks the **weighted average max level** reached as a function of days since install. Weighted average is used to account for varying player counts across level buckets. The percentage-change chart below highlights where the steepest level gains occur in the early-day window.

In [25]:
# hide-output
days_since_install_baseline = 0

data_filtered = data[data['days_since_install'] >= days_since_install_baseline]

pl_ftue_max_level_agg = compute_weighted_progression(data_filtered, measure_col='max_level', dimension_cols=['dummy', 'days_since_install','FTUE_flag','platform'], min_bucket_size=50)
pl_ftue_max_level_agg['combined_dimension'] = pl_ftue_max_level_agg['dummy'].astype(str) + ' | ' + pl_ftue_max_level_agg['FTUE_flag']

pl_ftue_max_level_agg['pctg_diff_max_level'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['weighted_avg_max_level'].pct_change().fillna(0)

pl_ftue_total_users = pl_ftue_max_level_agg[pl_ftue_max_level_agg['days_since_install'] == 0][['FTUE_flag', 'platform', 'cohort_users']].rename(columns={'cohort_users': 'total_cohort_users'})
pl_ftue_max_level_agg = pl_ftue_max_level_agg.merge(pl_ftue_total_users, on=['FTUE_flag','platform'], how='left')
pl_ftue_max_level_agg['pctg_users'] = pl_ftue_max_level_agg['cohort_users'] / pl_ftue_max_level_agg['total_cohort_users']
pl_ftue_max_level_agg['pctg_diff_users'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['pctg_users'].pct_change().fillna(0)

pl_ftue_max_level_agg


,dummy,days_since_install,FTUE_flag,platform,cohort_users,weighted_avg_max_level,combined_dimension,pctg_diff_max_level,total_cohort_users,pctg_users,pctg_diff_users
0,dummy,0,A.old,AND,5028,3.622040,dummy | A.old,0.000000,5028,1.000000,0.000000
1,dummy,0,A.old,IOS,11609,3.629166,dummy | A.old,0.000000,11609,1.000000,0.000000
2,dummy,0,B.new,AND,6713,3.655375,dummy | B.new,0.009203,6713,1.000000,0.000000
3,dummy,0,B.new,IOS,14814,3.662996,dummy | B.new,0.009322,14814,1.000000,0.000000
4,dummy,1,A.old,AND,2581,5.718138,dummy | A.old,0.000000,5028,0.513325,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
63,dummy,16,A.old,IOS,568,15.174877,dummy | A.old,0.000000,11609,0.048928,0.000000
64,dummy,16,B.new,IOS,433,15.413998,dummy | B.new,0.015758,14814,0.029229,-0.402604
65,dummy,17,A.old,IOS,332,15.338594,dummy | A.old,0.000000,11609,0.028599,0.000000
66,dummy,17,B.new,IOS,222,16.101302,dummy | B.new,0.049725,14814,0.014986,-0.475993


In [26]:
# hide-output
fig = px.bar(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='cohort_users',
              color='combined_dimension',
              title='Players at each day since install',
              facet_row='platform',
              width=1200,
              height=800,
              barmode='group',
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [28]:


fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='weighted_avg_max_level',
              color='combined_dimension',
              title='Player level reached at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

# hide-output
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='pctg_diff_max_level',
              color='combined_dimension',
              title='Player level reached at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [29]:
# hide-output
data.sort_values(['user_id', 'days_since_install'], inplace=True)
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
111122,10007491E7935123,2026-06-06,2026-06-06,2026-05-31,2026-06-01,0,6,3,IOS,CPE,0.76.0,Y,B.new,dummy
111292,10007491E7935123,2026-06-07,2026-06-06,2026-05-31,2026-06-01,1,6,4,IOS,CPE,0.76.0,Y,B.new,dummy
111995,10007491E7935123,2026-06-08,2026-06-06,2026-05-31,2026-06-01,2,7,4,IOS,CPE,0.76.0,Y,B.new,dummy
114190,10007491E7935123,2026-06-09,2026-06-06,2026-05-31,2026-06-01,3,7,4,IOS,CPE,0.76.0,Y,B.new,dummy
114354,10007491E7935123,2026-06-10,2026-06-06,2026-05-31,2026-06-01,4,8,5,IOS,CPE,0.76.0,Y,B.new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40492,FFFE079DACB6000A,2026-05-17,2026-05-16,2026-05-10,2026-05-01,1,6,4,IOS,UA,0.75.0,N,A.old,dummy
44255,FFFE079DACB6000A,2026-05-19,2026-05-16,2026-05-10,2026-05-01,3,8,5,IOS,UA,0.75.0,N,A.old,dummy
42074,FFFE079DACB6000A,2026-05-20,2026-05-16,2026-05-10,2026-05-01,4,8,5,IOS,UA,0.75.0,N,A.old,dummy
44312,FFFE079DACB6000A,2026-05-22,2026-05-16,2026-05-10,2026-05-01,6,8,5,IOS,UA,0.75.0,N,A.old,dummy


## Game day reached at day x (work in progress)

Mirrors the level analysis but uses **game days** (in-game calendar progression) instead of levels. Comparing both metrics reveals whether level gates or natural engagement drives pacing — if game days outpace levels, players are replaying content; if levels outpace game days, players are advancing quickly through fewer sessions.

In [42]:
# hide-output
# Same weighted-average approach as player level, applied to max_gameday

# Step 1: Count unique users per (install cohort, day since install, max_gameday bucket)
game_day_agg = data.groupby(['install_dt', 'days_since_install', 'max_gameday']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Share of each day's users at each game day value
game_day_agg = game_day_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
game_day_agg['percentage_of_daily_users'] = game_day_agg['unique_users'] / game_day_agg['total_unique_users']

# Step 4: Weighted average game day per cohort-day
weighted_avg = game_day_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_gameday'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_gameday']
game_day_agg = game_day_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small buckets (< 50 users) to reduce noise
game_day_agg = game_day_agg[game_day_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
game_day_agg = game_day_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_gameday = ('weighted_avg_max_gameday', 'first')
).reset_index()

game_day_agg

,install_dt,days_since_install,cohort_users,weighted_avg_max_gameday
0,2026-05-09,0,942,2.249478
1,2026-05-09,1,373,4.518595
2,2026-05-09,2,140,5.801609
3,2026-05-09,3,54,6.763804
4,2026-05-09,4,58,7.564626
...,...,...,...,...
249,2026-06-21,1,481,3.313758
250,2026-06-21,2,329,4.624473
251,2026-06-22,0,1056,2.039964
252,2026-06-22,1,465,3.325044


In [43]:
# hide-output
fig = px.line(game_day_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='install_dt',
              title='Game day daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},
              )


fig.show()

In [44]:
# hide-output
gameday_dist = data.groupby(['days_since_install', 'max_gameday']).agg(
    users=('user_id', 'count')
).reset_index()

gameday_pcts = gameday_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_gameday', include_groups=False).reset_index()

fig = px.line(
    gameday_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_gameday'),
    x='days_since_install',
    y='max_gameday',
    color='percentile',
    title='Game day distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

In [45]:
export_notebook_html(
# Exports the notebook to HTML for sharing and archival
    notebook_path='./earlyftue.ipynb',
    output_path='./earlyftue.html',
)

Saved to earlyftue.html


PosixPath('earlyftue.html')